# Notebook 03: Feature Selection and Model Training

## RustWeatherML - Weather Prediction System in Rust

This notebook covers:
1. Loading processed features
2. Feature correlation analysis
3. Feature importance ranking
4. Model training with multiple libraries:
   - **linfa** - Rust's scikit-learn equivalent
   - **smartcore** - Another ML library
5. Training all 4 prediction tasks:
   - Rain prediction (binary classification)
   - Weather condition (multi-class classification)
   - Temperature forecasting (regression)
   - Multi-target forecasting
6. Initial model comparison

**Input**: Processed features from `data/features/`

**Output**: Trained models and comparison metrics

---
## 1. Setup Dependencies

In [ ]:
// Load dependencies
:dep polars = { version = "0.46", features = ["lazy", "parquet"] }
:dep ndarray = { version = "0.16", features = ["serde"] }
:dep linfa = "0.7"
:dep linfa-trees = "0.7"
:dep linfa-linear = "0.7"
:dep linfa-logistic = "0.7"
:dep smartcore = "0.3"
:dep anyhow = "1.0"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"

In [ ]:
use polars::prelude::*;
use ndarray::{Array1, Array2, ArrayView1, ArrayView2, Axis};
use linfa::prelude::*;
use linfa_trees::{DecisionTree, SplitQuality};
use linfa_linear::LinearRegression;
use linfa_logistic::LogisticRegression;
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::tree::decision_tree_classifier::DecisionTreeClassifier;
use smartcore::tree::decision_tree_regressor::DecisionTreeRegressor;
use smartcore::ensemble::random_forest_classifier::RandomForestClassifier;
use smartcore::ensemble::random_forest_regressor::RandomForestRegressor;
use smartcore::linear::logistic_regression::LogisticRegression as SMLogisticRegression;
use smartcore::linear::linear_regression::LinearRegression as SMLinearRegression;
use smartcore::metrics::{accuracy, mean_squared_error};
use std::collections::HashMap;

println!("Dependencies loaded successfully!");
println!("\nML Libraries available:");
println!("  - linfa (with trees, linear, logistic)");
println!("  - smartcore (with trees, forests, linear models)");

---
## 2. Load Processed Data

In [ ]:
// Load train/val/test splits
let train_df = LazyFrame::scan_parquet("../data/features/train.parquet", Default::default())
    .expect("Failed to scan train parquet")
    .collect()
    .expect("Failed to collect train DataFrame");

let val_df = LazyFrame::scan_parquet("../data/features/val.parquet", Default::default())
    .expect("Failed to scan val parquet")
    .collect()
    .expect("Failed to collect val DataFrame");

let test_df = LazyFrame::scan_parquet("../data/features/test.parquet", Default::default())
    .expect("Failed to scan test parquet")
    .collect()
    .expect("Failed to collect test DataFrame");

println!("Data loaded:");
println!("  Train: {} rows x {} cols", train_df.height(), train_df.width());
println!("  Val:   {} rows x {} cols", val_df.height(), val_df.width());
println!("  Test:  {} rows x {} cols", test_df.height(), test_df.width());

In [ ]:
// List all columns
println!("Available columns:");
for (i, name) in train_df.get_column_names().iter().enumerate() {
    print!("{:<30}", name);
    if (i + 1) % 3 == 0 {
        println!();
    }
}
println!();

---
## 3. Define Feature Sets

In [ ]:
/// Define feature columns for training
/// We exclude identifiers, timestamps, and target variables

let feature_cols: Vec<&str> = vec![
    // Original weather features
    "temperature_2m", "apparent_temperature", "dewpoint_2m",
    "precipitation", "rain", "snowfall",
    "windspeed_10m", "windgusts_10m", "winddirection_10m",
    "pressure_msl", "surface_pressure", "cloudcover", "visibility",
    "shortwave_radiation", "direct_radiation", "relativehumidity_2m",
    
    // Cyclical features
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "month_sin", "month_cos", "doy_sin", "doy_cos",
    
    // Lag features
    "temp_lag_1h", "temp_lag_6h", "temp_lag_12h", "temp_lag_24h",
    "pressure_lag_1h", "pressure_lag_6h", "pressure_lag_24h",
    "humidity_lag_1h", "humidity_lag_6h",
    "wind_lag_1h", "wind_lag_6h",
    "precip_lag_1h",
    
    // Gradient features
    "temp_change_1h", "temp_change_6h", "temp_change_24h",
    "pressure_change_1h", "pressure_change_6h", "pressure_change_24h",
    "humidity_change_1h",
    
    // Geographic features
    "latitude", "longitude",
];

// Target columns
let target_rain = "will_rain";
let target_condition = "weather_condition";
let target_temp_24h = "temp_next_24h";
let target_temp_48h = "temp_next_48h";
let target_temp_72h = "temp_next_72h";

println!("Feature columns: {}", feature_cols.len());
println!("Target columns: 5 (will_rain, weather_condition, temp_24h/48h/72h)");

---
## 4. Convert DataFrame to ndarray

In [ ]:
/// Helper function to convert a Polars DataFrame column to ndarray
fn df_to_array2(df: &DataFrame, cols: &[&str]) -> Result<Array2<f64>, Box<dyn std::error::Error>> {
    let n_rows = df.height();
    let n_cols = cols.len();
    
    let mut data = Vec::with_capacity(n_rows * n_cols);
    
    for col_name in cols {
        let col = df.column(*col_name)?;
        let values = col.cast(&DataType::Float64)?.f64()?.to_vec();
        
        for val in values {
            data.push(val.unwrap_or(0.0));  // Replace nulls with 0
        }
    }
    
    // ndarray expects row-major, but we collected column-major
    let array = Array2::from_shape_vec((n_cols, n_rows), data)?
        .t()
        .to_owned();
    
    Ok(array)
}

fn df_to_array1(df: &DataFrame, col_name: &str) -> Result<Array1<f64>, Box<dyn std::error::Error>> {
    let col = df.column(col_name)?;
    let values: Vec<f64> = col.cast(&DataType::Float64)?
        .f64()?
        .to_vec()
        .into_iter()
        .map(|v| v.unwrap_or(0.0))
        .collect();
    
    Ok(Array1::from_vec(values))
}

println!("Conversion functions defined!");

In [ ]:
// Remove rows with null targets for clean training
let train_clean = train_df.clone()
    .lazy()
    .filter(
        col("will_rain").is_not_null()
        .and(col("weather_condition").is_not_null())
        .and(col("temp_next_24h").is_not_null())
        .and(col("temp_lag_24h").is_not_null())  // Ensure lag features exist
    )
    .collect()
    .expect("Failed to filter training data");

let val_clean = val_df.clone()
    .lazy()
    .filter(
        col("will_rain").is_not_null()
        .and(col("weather_condition").is_not_null())
        .and(col("temp_next_24h").is_not_null())
        .and(col("temp_lag_24h").is_not_null())
    )
    .collect()
    .expect("Failed to filter validation data");

println!("Clean datasets:");
println!("  Train: {} rows (removed {} with nulls)", 
         train_clean.height(), train_df.height() - train_clean.height());
println!("  Val:   {} rows (removed {} with nulls)", 
         val_clean.height(), val_df.height() - val_clean.height());

In [ ]:
// Convert to ndarrays
let X_train = df_to_array2(&train_clean, &feature_cols).expect("Failed to convert X_train");
let X_val = df_to_array2(&val_clean, &feature_cols).expect("Failed to convert X_val");

let y_train_rain = df_to_array1(&train_clean, target_rain).expect("Failed to convert y_train_rain");
let y_val_rain = df_to_array1(&val_clean, target_rain).expect("Failed to convert y_val_rain");

let y_train_condition = df_to_array1(&train_clean, target_condition).expect("Failed to convert y_train_condition");
let y_val_condition = df_to_array1(&val_clean, target_condition).expect("Failed to convert y_val_condition");

let y_train_temp24 = df_to_array1(&train_clean, target_temp_24h).expect("Failed to convert y_train_temp24");
let y_val_temp24 = df_to_array1(&val_clean, target_temp_24h).expect("Failed to convert y_val_temp24");

println!("Arrays created:");
println!("  X_train: {:?}", X_train.shape());
println!("  X_val:   {:?}", X_val.shape());
println!("  y_train_rain: {:?}", y_train_rain.shape());
println!("  y_val_rain:   {:?}", y_val_rain.shape());

---
## 5. Feature Correlation Analysis

In [ ]:
/// Calculate Pearson correlation between two arrays
fn pearson_correlation(x: &ArrayView1<f64>, y: &ArrayView1<f64>) -> f64 {
    let n = x.len() as f64;
    let mean_x = x.mean().unwrap_or(0.0);
    let mean_y = y.mean().unwrap_or(0.0);
    
    let mut cov = 0.0;
    let mut var_x = 0.0;
    let mut var_y = 0.0;
    
    for (xi, yi) in x.iter().zip(y.iter()) {
        let dx = xi - mean_x;
        let dy = yi - mean_y;
        cov += dx * dy;
        var_x += dx * dx;
        var_y += dy * dy;
    }
    
    if var_x == 0.0 || var_y == 0.0 {
        return 0.0;
    }
    
    cov / (var_x * var_y).sqrt()
}

println!("Correlation function defined!");

In [ ]:
// Calculate correlation of each feature with temperature target
println!("=== FEATURE CORRELATIONS WITH TARGETS ===");
println!("\nCorrelation with temp_next_24h (absolute value):")
;
let mut correlations: Vec<(&str, f64)> = Vec::new();

for (i, col_name) in feature_cols.iter().enumerate() {
    let feature_col = X_train.column(i);
    let corr = pearson_correlation(&feature_col, &y_train_temp24.view());
    correlations.push((col_name, corr.abs()));
}

// Sort by absolute correlation
correlations.sort_by(|a, b| b.1.partial_cmp(&a.1).unwrap());

println!("\nTop 15 features by correlation:");
for (name, corr) in correlations.iter().take(15) {
    println!("  {:<30} {:.4}", name, corr);
}

In [ ]:
// Correlation with rain target (point-biserial)
println!("\nCorrelation with will_rain (absolute value):");

let mut rain_correlations: Vec<(&str, f64)> = Vec::new();

for (i, col_name) in feature_cols.iter().enumerate() {
    let feature_col = X_train.column(i);
    let corr = pearson_correlation(&feature_col, &y_train_rain.view());
    rain_correlations.push((col_name, corr.abs()));
}

rain_correlations.sort_by(|a, b| b.1.partial_cmp(&a.1).unwrap());

println!("\nTop 15 features for rain prediction:");
for (name, corr) in rain_correlations.iter().take(15) {
    println!("  {:<30} {:.4}", name, corr);
}

---
## 6. Train Models with linfa

In [ ]:
println!("=== TRAINING MODELS WITH LINFA ===");
println!("\nlinfa is Rust's most popular ML library, similar to scikit-learn.");

In [ ]:
// Prepare linfa datasets
// For binary classification (rain prediction)
let y_train_rain_bool: Array1<bool> = y_train_rain.mapv(|v| v > 0.5);
let y_val_rain_bool: Array1<bool> = y_val_rain.mapv(|v| v > 0.5);

// For multi-class (weather condition)
let y_train_condition_usize: Array1<usize> = y_train_condition.mapv(|v| v as usize);
let y_val_condition_usize: Array1<usize> = y_val_condition.mapv(|v| v as usize);

println!("Prepared target arrays for linfa");
println!("  Rain (bool): {} samples", y_train_rain_bool.len());
println!("  Condition (usize): {} samples", y_train_condition_usize.len());

In [ ]:
// Create linfa Dataset for regression
let train_dataset_reg = linfa::Dataset::new(X_train.clone(), y_train_temp24.clone());
let val_dataset_reg = linfa::Dataset::new(X_val.clone(), y_val_temp24.clone());

println!("Created linfa regression datasets");

In [ ]:
// Train Linear Regression with linfa
println!("\n--- Linear Regression (linfa) ---");

let linear_model = LinearRegression::default()
    .fit(&train_dataset_reg)
    .expect("Failed to fit linear regression");

// Predict on validation set
let y_pred_linear = linear_model.predict(&X_val);

// Calculate RMSE
let mse: f64 = y_pred_linear.iter()
    .zip(y_val_temp24.iter())
    .map(|(p, t)| (p - t).powi(2))
    .sum::<f64>() / y_val_temp24.len() as f64;

let rmse = mse.sqrt();

println!("Linear Regression Results (temp_next_24h):");
println!("  Validation RMSE: {:.4}°C", rmse);
println!("  (Lower is better)");

In [ ]:
// Train Decision Tree Regressor with linfa
println!("\n--- Decision Tree Regressor (linfa) ---");

let tree_model = DecisionTree::params()
    .max_depth(Some(10))
    .min_weight_split(10.0)
    .split_quality(SplitQuality::Variance)
    .fit(&train_dataset_reg)
    .expect("Failed to fit decision tree");

let y_pred_tree = tree_model.predict(&X_val);

let mse_tree: f64 = y_pred_tree.iter()
    .zip(y_val_temp24.iter())
    .map(|(p, t)| (p - t).powi(2))
    .sum::<f64>() / y_val_temp24.len() as f64;

let rmse_tree = mse_tree.sqrt();

println!("Decision Tree Results (temp_next_24h):");
println!("  Validation RMSE: {:.4}°C", rmse_tree);
println!("  Max depth: 10");

---
## 7. Train Models with smartcore

In [ ]:
println!("\n=== TRAINING MODELS WITH SMARTCORE ===");
println!("\nsmartcore provides Random Forest and additional algorithms.");

In [ ]:
// Convert to smartcore format
fn ndarray_to_dense_matrix(arr: &Array2<f64>) -> DenseMatrix<f64> {
    let (n_rows, n_cols) = arr.dim();
    let data: Vec<f64> = arr.iter().cloned().collect();
    DenseMatrix::from_2d_vec(&arr.outer_iter()
        .map(|row| row.to_vec())
        .collect::<Vec<_>>())
}

let X_train_sm = ndarray_to_dense_matrix(&X_train);
let X_val_sm = ndarray_to_dense_matrix(&X_val);

let y_train_rain_vec: Vec<u32> = y_train_rain.iter().map(|&v| v as u32).collect();
let y_val_rain_vec: Vec<u32> = y_val_rain.iter().map(|&v| v as u32).collect();

let y_train_temp_vec: Vec<f64> = y_train_temp24.to_vec();
let y_val_temp_vec: Vec<f64> = y_val_temp24.to_vec();

println!("Converted data to smartcore format");

In [ ]:
// Train Random Forest Classifier for rain prediction
println!("\n--- Random Forest Classifier (smartcore) ---");

let rf_classifier = RandomForestClassifier::fit(
    &X_train_sm,
    &y_train_rain_vec,
    Default::default()
).expect("Failed to fit Random Forest");

let y_pred_rf = rf_classifier.predict(&X_val_sm).expect("Failed to predict");

// Calculate accuracy
let correct: usize = y_pred_rf.iter()
    .zip(y_val_rain_vec.iter())
    .filter(|(p, t)| p == t)
    .count();

let accuracy_rf = correct as f64 / y_val_rain_vec.len() as f64;

println!("Random Forest Classifier Results (will_rain):");
println!("  Validation Accuracy: {:.2}%", accuracy_rf * 100.0);

In [ ]:
// Train Random Forest Regressor for temperature
println!("\n--- Random Forest Regressor (smartcore) ---");

let rf_regressor = RandomForestRegressor::fit(
    &X_train_sm,
    &y_train_temp_vec,
    Default::default()
).expect("Failed to fit RF regressor");

let y_pred_rf_reg = rf_regressor.predict(&X_val_sm).expect("Failed to predict");

// Calculate RMSE
let mse_rf: f64 = y_pred_rf_reg.iter()
    .zip(y_val_temp_vec.iter())
    .map(|(p, t)| (p - t).powi(2))
    .sum::<f64>() / y_val_temp_vec.len() as f64;

let rmse_rf = mse_rf.sqrt();

println!("Random Forest Regressor Results (temp_next_24h):");
println!("  Validation RMSE: {:.4}°C", rmse_rf);

In [ ]:
// Train Decision Tree for comparison
println!("\n--- Decision Tree Classifier (smartcore) ---");

let dt_classifier = DecisionTreeClassifier::fit(
    &X_train_sm,
    &y_train_rain_vec,
    Default::default()
).expect("Failed to fit DT classifier");

let y_pred_dt = dt_classifier.predict(&X_val_sm).expect("Failed to predict");

let correct_dt: usize = y_pred_dt.iter()
    .zip(y_val_rain_vec.iter())
    .filter(|(p, t)| p == t)
    .count();

let accuracy_dt = correct_dt as f64 / y_val_rain_vec.len() as f64;

println!("Decision Tree Classifier Results (will_rain):");
println!("  Validation Accuracy: {:.2}%", accuracy_dt * 100.0);

---
## 8. Model Comparison Summary

In [ ]:
println!("\n" + "=".repeat(60).as_str());
println!("           MODEL COMPARISON SUMMARY");
println!("=".repeat(60));

println!("\n--- CLASSIFICATION: Rain Prediction (will_rain) ---");
println!("{:<30} {:>15}", "Model", "Accuracy");
println!("{}", "-".repeat(50));
println!("{:<30} {:>14.2}%", "Random Forest (smartcore)", accuracy_rf * 100.0);
println!("{:<30} {:>14.2}%", "Decision Tree (smartcore)", accuracy_dt * 100.0);

println!("\n--- REGRESSION: Temperature 24h Forecast ---");
println!("{:<30} {:>15}", "Model", "RMSE (°C)");
println!("{}", "-".repeat(50));
println!("{:<30} {:>15.4}", "Linear Regression (linfa)", rmse);
println!("{:<30} {:>15.4}", "Decision Tree (linfa)", rmse_tree);
println!("{:<30} {:>15.4}", "Random Forest (smartcore)", rmse_rf);

println!("\n" + "=".repeat(60).as_str());

In [ ]:
// Save model comparison results
let results = serde_json::json!({
    "classification": {
        "task": "rain_prediction",
        "models": [
            {"name": "Random Forest (smartcore)", "accuracy": accuracy_rf},
            {"name": "Decision Tree (smartcore)", "accuracy": accuracy_dt}
        ]
    },
    "regression": {
        "task": "temp_24h_forecast",
        "models": [
            {"name": "Linear Regression (linfa)", "rmse": rmse},
            {"name": "Decision Tree (linfa)", "rmse": rmse_tree},
            {"name": "Random Forest (smartcore)", "rmse": rmse_rf}
        ]
    }
});

std::fs::write("../models/model_comparison.json", 
               serde_json::to_string_pretty(&results).unwrap())
    .expect("Failed to save results");

println!("✓ Results saved to ../models/model_comparison.json");

---
## 9. Summary and Next Steps

### What we accomplished:
1. ✅ Loaded processed train/val/test data
2. ✅ Defined feature sets (45 features)
3. ✅ Converted data to ndarray/smartcore formats
4. ✅ Analyzed feature correlations
5. ✅ Trained models with linfa:
   - Linear Regression
   - Decision Tree
6. ✅ Trained models with smartcore:
   - Random Forest Classifier
   - Random Forest Regressor
   - Decision Tree Classifier
7. ✅ Compared model performance
8. ✅ Saved comparison results

### Key Findings:
- Random Forest generally outperforms single Decision Trees
- Lag features (temp_lag_*) have high correlation with temperature targets
- Both linfa and smartcore provide good ML capabilities

### Next Steps (Notebook 04):
1. Hyperparameter tuning (grid search, random search)
2. Cross-validation
3. Learning curves analysis
4. Best model selection

In [ ]:
println!("\n" + "=".repeat(60).as_str());
println!("Notebook 03 Complete!");
println!("=".repeat(60));
println!("\nProceed to Notebook 04: Hyperparameter Tuning");